# CNN Practice: Define a Model and Inspect Its Outputs

## 1. Model Definition

Define the model before preparing its input. The feature block creates 16 maps of 14 × 14 from each 28 × 28 greyscale image. Flatten joins the maps into 3,136 values, and the final Linear layer produces ten class scores.

This is the same `DigitCNN` class used in Section 2 of the lesson. During testing, inspect the layers individually and compare their final result with a complete model call. The weights start randomly; this notebook does not train the model.


In [ ]:
import torch
from torch import nn

class DigitCNN(nn.Module):
    """Extract image features and produce ten digit-class scores."""

    def __init__(self):
        super().__init__()  # Initialise the parent module before adding layers.
        self.features = nn.Sequential(
            ####### finish the feature block with a convolutional layer, ReLU activation, and max pooling layer
            
        )
        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),  # Flatten the maps while keeping the batch axis.
            nn.Linear(in_features=16 * 14 * 14, out_features=10),  # Produce ten class scores directly from the flattened features.
        )

    def forward(self, images):
        features = self.features(images)  # Extract and pool local image features.
        return self.classifier(features)  # Convert the features into ten class scores.

torch.manual_seed(42)  # Reproduce the initial random model weights.
model = DigitCNN()  # Create the same feature block and classifier as the lesson.
print(model)
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))
print('Weights are randomly initialised; this model has not been trained.')


## 2. Image Input

Set `TEST_IMAGE_COUNT` to choose how many digits to test. The default is **1**; change it to a larger number and rerun the following cells to test and display multiple images.

Read that many records from the CSV. Each row contains a label followed by 784 pixels, which form a 28 × 28 greyscale image.


In [ ]:
import numpy as np

DATA_PATH = 'sample_data/mnist_train_small.csv'
IMAGE_SIZE = 28
TEST_IMAGE_COUNT = 1  # Change this to test more images, for example 5.

assert TEST_IMAGE_COUNT >= 1, 'Choose at least one test image.'
values = np.loadtxt(
    DATA_PATH, delimiter=',', max_rows=TEST_IMAGE_COUNT,
    dtype=np.float32, ndmin=2,  # Keep a row dimension when reading just one image.
)
assert len(values) == TEST_IMAGE_COUNT, 'The CSV contains fewer images than requested.'
labels = values[:, 0].astype(np.int64)
# Restore each row of pixels to a 28 × 28 greyscale image.
images = values[:, 1:].astype(np.uint8).reshape(-1, IMAGE_SIZE, IMAGE_SIZE)
print('Images:', images.shape, images.dtype)
print('Labels:', labels.tolist())


In [ ]:
import matplotlib.pyplot as plt

plot_columns = min(TEST_IMAGE_COUNT, 4)  # Display at most four images per row.
plot_rows = (TEST_IMAGE_COUNT + plot_columns - 1) // plot_columns
figure, axes = plt.subplots(
    plot_rows, plot_columns, figsize=(3 * plot_columns, 3 * plot_rows),
    squeeze=False,  # Keep a two-dimensional axes array even for one image.
)
for image_index, axis in enumerate(axes.ravel()):
    axis.axis('off')
    if image_index < TEST_IMAGE_COUNT:
        axis.imshow(images[image_index], cmap='gray', vmin=0, vmax=255)  # Use the full pixel range.
        axis.set_title(f'Image {image_index + 1}: digit {labels[image_index]}')
figure.tight_layout()  # Keep image titles separate.
plt.show()


Convert every selected image to a floating-point tensor and normalise its values to −1–1. Stack the tensors into `[N, 1, 28, 28]`, where `N` is `TEST_IMAGE_COUNT`. The leading dimension remains present when testing one image.


In [ ]:
from PIL import Image
from torchvision import transforms

NORMALISATION_MEAN = (0.5,)  # Supply one mean for the greyscale channel.
NORMALISATION_STD = (0.5,)  # Map [0, 1] to [-1, 1].

image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),  # Set exact height and width.
    transforms.ToTensor(),  # Convert uint8 pixels to CHW floats in [0, 1].
    transforms.Normalize(mean=NORMALISATION_MEAN, std=NORMALISATION_STD),
])
# Transform every selected image and retain the image-count dimension.
test_images = torch.stack([image_transform(Image.fromarray(image)) for image in images])
print('Model input:', tuple(test_images.shape), test_images.dtype)
for image_index, tensor in enumerate(test_images):
    print(f'Image {image_index + 1}: range {tensor.min().item():.2f} to {tensor.max().item():.2f}')


## 3. Model Testing and Output Results

Run the selected images through the model in evaluation mode without recording gradients. Inspect each layer's shape and a small sample of its values **for every image**, then display all images with their actual and predicted classes.

| Stage | Expected shape (`N` = number of test images) |
|---|---|
| Input | `[N, 1, 28, 28]` |
| Conv2d | `[N, 16, 28, 28]` |
| ReLU | `[N, 16, 28, 28]` |
| MaxPool2d | `[N, 16, 14, 14]` |
| Flatten | `[N, 3136]` |
| Linear | `[N, 10]` |


In [ ]:
PATCH_SIZE = 5
VECTOR_PREVIEW = 10
torch.set_printoptions(precision=4, sci_mode=False)  # Round printed values without changing tensors.

model_device = next(model.parameters()).device
test_images = test_images.to(model_device)  # Match the model device, including after training.
model.eval()  # Select evaluation behaviour.
with torch.no_grad():  # Inspect the model without recording gradients.
    logits = model(test_images)  # Return ten class scores for each selected image.
    layer_outputs = {'Input': test_images}
    features = test_images
    layer_names = ['Conv2d', 'ReLU', 'MaxPool2d', 'Flatten', 'Linear']
    layers = list(model.features) + list(model.classifier)
    for layer_name, layer in zip(layer_names, layers):
        features = layer(features)  # Inspect the layers in forward-pass order.
        layer_outputs[layer_name] = features
    assert torch.equal(layer_outputs['Linear'], logits)

for layer_name, output in layer_outputs.items():
    print(f'\n{layer_name}: shape={tuple(output.shape)}, dtype={output.dtype}')
    for image_index, image_output in enumerate(output):
        print(f'Image {image_index + 1}: min={image_output.min().item():.4f}, max={image_output.max().item():.4f}')
        if output.ndim == 4:
            print('First channel, top-left patch:')
            print(image_output[0, :PATCH_SIZE, :PATCH_SIZE])
        else:
            print('First vector values:')
            print(image_output[:VECTOR_PREVIEW])

assert layer_outputs['Conv2d'].shape == (TEST_IMAGE_COUNT, 16, 28, 28)
assert layer_outputs['MaxPool2d'].shape == (TEST_IMAGE_COUNT, 16, 14, 14)
assert layer_outputs['Flatten'].shape == (TEST_IMAGE_COUNT, 3136)
assert logits.shape == (TEST_IMAGE_COUNT, 10)
predictions = logits.argmax(dim=1).tolist()  # Choose the highest-scoring class for each image.
for image_index, (actual, predicted) in enumerate(zip(labels.tolist(), predictions)):
    print(f'Image {image_index + 1}: actual={actual}, untrained prediction={predicted}')

figure, axes = plt.subplots(
    plot_rows, plot_columns, figsize=(3 * plot_columns, 3 * plot_rows), squeeze=False,
)
for image_index, axis in enumerate(axes.ravel()):
    axis.axis('off')
    if image_index < TEST_IMAGE_COUNT:
        axis.imshow(images[image_index], cmap='gray', vmin=0, vmax=255)
        axis.set_title(f'Image {image_index + 1}\nActual: {labels[image_index]} | Predicted: {predictions[image_index]}')
figure.suptitle('Untrained model predictions')
figure.tight_layout()  # Keep all image titles visible.
plt.show()


## 4. Train the Model

After running the cells above, copy the code cells from **Sections 3.2, 3.3 and 3.4** of [Session 12: Building and Training CNN Models with PyTorch](<Session12 Building and Training CNN Models with PyTorch.ipynb>) into new code cells below, in that order.

- Section 3.2 loads and splits the digit data, using the same −1–1 normalisation as this notebook.
- Section 3.3 moves the existing `model` to the training device and creates the loss function and optimiser.
- Section 3.4 defines the training helper, trains for one epoch and prints training loss and accuracy.

The predictions in Section 3 above show the randomly initialised model. Rerun Section 3 to inspect predictions after training; its code places the input on the model's device. For validation, learning curves and test evaluation, continue with Sections 3.5–4 of Session 12.
